# 1 Importación de librerías

In [24]:
import pandas as pd
import glob
import os
import warnings
warnings.filterwarnings('ignore')

# 2 Carga de datos y unión

In [26]:
path = os.path.join('..', 'data', 'raw') 

# Buscamos y unimos los 12 archivos CSV
all_files = glob.glob(os.path.join(path, "PRSA_Data_*.csv"))

li = [pd.read_csv(filename) for filename in all_files]
df_raw = pd.concat(li, axis=0, ignore_index=True)

print(f"Archivos cargados: {len(li)}")
print(f"Total de registros: {len(df_raw)}")

Archivos cargados: 12
Total de registros: 420768


# 3 Transformación y limpieza de datos

En esta fase:
1. Unificamos las columnas temporales en un índice `date`.
2. Utilizamos interpolación lineal agrupada por estación para no perder información.
3. Rellenamos hacia adelante (`ffill`) la variable categórica de la dirección del viento (`wd`).

In [29]:
df_clean = df_raw.copy()

# 1. Tratamiento de fechas
df_clean['date'] = pd.to_datetime(df_clean[['year', 'month', 'day', 'hour']])
df_clean = df_clean.sort_values(by=['station', 'date'])
df_clean.set_index('date', inplace=True)
df_clean.drop(columns=['No', 'year', 'month', 'day', 'hour'], inplace=True, errors='ignore')

# 2. Interpolación lineal para variables numéricas (agrupando por estación)
cols_a_limpiar = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
df_clean[cols_a_limpiar] = df_clean.groupby('station')[cols_a_limpiar].transform(lambda x: x.interpolate(method='linear'))

# 3. Relleno de la dirección del viento (categórica)
df_clean['wd'] = df_clean.groupby('station')['wd'].ffill()
df_clean['wd'] = df_clean['wd'].bfill() # Por si queda algún nulo residual al inicio

# 4. Eliminación de filas que aún puedan contener nulos tras la interpolación
df_clean = df_clean.dropna()

print("Valores nulos restantes por columna:")
print(df_clean.isnull().sum())
df_clean.head()

Valores nulos restantes por columna:
PM2.5      0
PM10       0
SO2        0
NO2        0
CO         0
O3         0
TEMP       0
PRES       0
DEWP       0
RAIN       0
wd         0
WSPM       0
station    0
dtype: int64


,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
date,,,,,,,,,,,,,
2013-03-01 00:00:00,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
2013-03-01 01:00:00,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2013-03-01 02:00:00,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
2013-03-01 03:00:00,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
2013-03-01 04:00:00,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


# 4 Guardar la tabla unificada y limpia

In [31]:
ruta_carpeta = '../data/processed'
nombre_archivo = 'beijing_unified_cleaned.csv'
ruta_completa = os.path.join(ruta_carpeta, nombre_archivo)

os.makedirs(ruta_carpeta, exist_ok=True)

df_clean.to_csv(ruta_completa, index=True)
print(f"✅ Dataset unificado y limpio guardado exitosamente en: {ruta_completa}")

✅ Dataset unificado y limpio guardado exitosamente en: ../data/processed\beijing_unified_cleaned.csv
